# 02 Feature Engineering

Build indicators, daily context, forward labels, trailing shifted rolling z-scores, lagged LightGBM features, processed Parquet data, and `feature_config.json`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import subprocess
import sys

BASE = Path('/content/drive/MyDrive/trading_system')
REPO_DIR = BASE / 'TradingBot26'
DATA_DIR = BASE / 'data'
MODEL_DIR = BASE / 'models'
LOG_DIR = BASE / 'logs'
REPORT_DIR = BASE / 'reports'
ARTIFACT_DIR = BASE / 'artifacts'

%cd $REPO_DIR
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)])

In [ ]:
import json

from config import FeatureConfig, LabelConfig, MarketConfig, NormalizerConfig, PathConfig
from data.loader import MarketDataLoader
from features.pipeline import FeatureEngineeringPipeline

paths = PathConfig(
    root=BASE,
    raw_data_dir=DATA_DIR / 'raw',
    processed_data_dir=DATA_DIR / 'processed',
    artifact_dir=ARTIFACT_DIR,
    model_artifact_dir=MODEL_DIR,
    report_dir=REPORT_DIR,
)
if (DATA_DIR / 'raw' / 'BANKNIFTY_5m_intraday.parquet').exists():
    market = MarketConfig(symbol='BANKNIFTY', ticker='BANKNIFTY', series='EQ', interval='5m', intraday_source='drive-cache-local-csv', daily_source='intraday-resample')
elif (DATA_DIR / 'raw' / 'NIFTY_5m_intraday.parquet').exists():
    market = MarketConfig(symbol='NIFTY', ticker='NIFTY', series='EQ', interval='5m', intraday_source='drive-cache-local-csv', daily_source='intraday-resample')
else:
    market = MarketConfig(symbol='RELIANCE', ticker='RELIANCE.NS', series='EQ', interval='5m', intraday_source='drive-cache-or-jugaad-openchart')
labels = LabelConfig(horizon=15, target_pct=0.005, stop_pct=0.003)
normalizer = NormalizerConfig(window=200, min_periods=200)
features = FeatureConfig(lag_periods=(1, 5, 10), volume_confirm_window=20, include_daily_context=True)

loader = MarketDataLoader(paths, market)
pipeline = FeatureEngineeringPipeline(paths=paths, market=market, features=features, labels=labels, normalizer=normalizer)
dataset = pipeline.run(loader.load_intraday(), loader.load_daily_context())

print(f'Processed rows: {len(dataset.frame):,}')
print(f'Feature columns: {len(dataset.feature_columns):,}')
print(f'Feature config: {dataset.feature_config_path}')
print(dataset.frame['label_name'].value_counts(dropna=False))

metadata = json.loads(dataset.feature_config_path.read_text())
metadata['feature_columns'][:10], metadata['normalization'], metadata['labels']